In [ ]:
from PIL import Image
import math
import MiniNumPy as mnp

In [40]:
# load image and convert to 3D matrix (h, w, 3)
def load_image(path):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    pixels = list(img.getdata())

    matrix = []
    for i in range(h):
        row = []
        for j in range(w):
            row.append(list(pixels[i*w + j]))  # [R, G, B]
        matrix.append(row)

    return mnp.array(matrix)

# save 3D matrix as image
def save_image(mat, path):
    h, w = mat.shape[0], mat.shape[1]

    img = Image.new("RGB", (w, h))
    flat = []

    for i in range(h):
        for j in range(w):
            pixel = mat.data[i][j]
            pixel = tuple(int(x) for x in pixel) 
            flat.append(pixel)

    img.putdata(flat)
    img.save(path)

## Scaling Image 

In [41]:
def scale_matrix(sx, sy):
    return mnp.array([
        [sx, 0],
        [0, sy]
    ])


In [42]:
# 2-D image sample 

points = mnp. array([
    [0, 0],
    [3, 0],
    [1, 3],
    [0, 5]
])

S = scale_matrix(5,5)
rel = points @ S
print(rel)

[[0 0]
 [15 0]
 [5 15]
 [0 25]]


## Rotating matrix

In [43]:
def rotation_matrix(theta):
    return mnp.array([
        [math.cos(theta), -math.sin(theta)],
        [math.sin(theta),  math.cos(theta)]
    ])
    
def rotate_image(img , theta):
    H,W,_  = img.shape # h,w,3 
    cx, cy = W//2, H//2 #center coordinate
    
    R = rotation_matrix(theta)
    
    new_img = []
    for y in range(H):
        row = []
        for x in range(W):
            
            v = mnp.array([[x - cx],[y - cy]]) # relative coordinate related to center
            new = R @ v
            
            # to the world frame coordinate
            nx = int(new.data[0][0] + cx)
            ny = int(new.data[1][0] + cy)
            
            # if the new pixel coordinate in valid range, assign the color to that pixel
            if 0 <= nx < W and 0< ny < H:
                row.append(img.data[ny][nx])
            else:
                row.append([0, 0, 0])
        new_img.append(row)
    
    return mnp.array(new_img)

In [44]:
img = load_image("hawksbill_sea_turtle.jpg") 
print(img.shape)
rot = rotate_image(img, math.radians(30))
save_image(rot, "rotated_image.png")

(500, 750, 3)


## Manipulating Color Channels

In [52]:
# convert image to grayscale    
def to_grayscale(img):
    H, W, _ = img.shape
    gray = []

    for i in range(H):
        row = []
        for j in range(W):
            R, G, B = img.data[i][j]
            # gray value
            g = 0.299*R + 0.587*G + 0.114*B
            row.append([g, g, g])
        gray.append(row)

    return mnp.array(gray)

# Increase the brightness of the red channel

def more_red(img, factor = 1.5):
    H, W, _ = img.shape
    red = []

    for i in range(H):
        row = []
        for j in range(W):
            R, G, B = img.data[i][j]
            # more red
            new_red = R * factor
            new_red = min(255, max(0, new_red))
            
            row.append([new_red, G, B])
        red.append(row)

    return mnp.array(red)

# Adjust the contrast of the image by multiplying each channel by a constant
def mo_con(img):
    H, W, _ = img.shape
    con = []

    for i in range(H):
        row = []
        for j in range(W):
            R, G, B = img.data[i][j]
            # contrast
            g = 0.9*R + 0.6*G + 0.3*B
            row.append([g, g, g])
        con.append(row)

    return mnp.array(con)

In [53]:
gray = to_grayscale(img)
save_image(gray, "gray.jpg")

red = more_red(img, 2)
save_image(red, "red.jpg")

con = mo_con(img)
save_image(con, "con.jpg")

## Translation of the Image

In [47]:
def translational_matrix(tx, ty):
    return mnp.array([
        [1, 0 , tx],
        [0, 1,  ty],
        [0, 0, 1]
    ])

def translational_image(img, tx, ty):
    H, W, _ = img.shape
    cx, cy = W // 2, H // 2

    T = translational_matrix(tx, ty)

    new_img = []

    for y in range(H):
        row = []
        for x in range(W):

           # relative coordinate related to center
            v = mnp.array([[x - cx], [y - cy], [1]])

            # apply trans transform
            new = T @ v
            
            # to the world frame coordinate
            nx = int(new.data[0][0] + cx)
            ny = int(new.data[1][0] + cy)

            if 0 <= nx < W and 0 <= ny < H:
                row.append(img.data[ny][nx])
            else:
                row.append([0, 0, 0])

        new_img.append(row)

    return mnp.array(new_img)

In [48]:
trans = translational_image(img, 100, 100)
save_image(trans, "trans.png")